# FinQA 数据先导复现

目的：学习数据粒度、标签引用检查和完整证据覆盖；不训练模型、不访问测试集。

已执行脚本记录：6251道训练题、883道开发题；页面交集为0，公司—年度代理组交集197。固定词法先导Train 18/30、Dev 26/30完整Coverage@5。

本notebook尚未用Jupyter内核运行；底层脚本已运行。先从项目根目录运行 `python research/ingest/fetch_finqa.py` 获取允许的数据。

## 1. 定位项目与固定数据版本

从项目目录启动notebook；公司—年度ID只是源ID解析的分组代理。原始数据和大产物不提交到Git。

In [ ]:
from pathlib import Path
import sys, json
start = Path.cwd().resolve()
project = next((p for p in (start, *start.parents) if (p / "research/evaluation/audit_finqa.py").is_file()), None)
if project is None:
    raise RuntimeError("请从 financial-risk-platform 项目目录启动 notebook")
sys.path.insert(0, str(project / "research/evaluation"))
from audit_finqa import audit, COMMIT
data = project / "research/data/finqa" / COMMIT
output = project / "research/artifacts/finqa-notebook-audit"
if not (data / "acquisition.json").is_file():
    raise RuntimeError("请先运行 research/ingest/fetch_finqa.py")
print("固定版本：", COMMIT)


## 2. 重跑结构与标签审计

只读取train/dev；候选构造不读取答案或model_input。完整标注集无法全部召回时仍记失败，不向候选补gold。

In [ ]:
result = audit(data, output)
summary = {split: {"questions": p["questions"], "groups": p["groups"], "problems": p["problems"]} for split, p in result["profiles"].items()}
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("Train/Dev交集：", json.dumps(result["train_dev_overlap_counts"], ensure_ascii=False))
assert result["test_content_accessed"] is False


## 3. 查看60题词法诊断

同一页内BM25；30题/划分；这些比例不是最终测试准确率，也不证明中文或跨文档能力。

In [ ]:
for split, p in result["pilot"].items():
    print(split, "完整Coverage@5：", p["complete_gold_coverage_at_5_count"], "/", p["questions"])
    print(split, "候选完整Coverage@20：", p["candidate_gold_coverage_at_20_count"], "/", p["questions"])


## 4. 你可以自己核对的问题

- 为什么页面不重叠仍不能证明未见年度报告泛化？
- 为什么text_-1不能直接用Python负索引映射？
- 为什么标点空格不同不能直接判财务内容错误？
- 为什么未标为gold的候选不能全部当确认负例？
- 为什么需要同时看top20候选上限和top5完整覆盖？

个人回答与练习完成情况不预填。详细发现见 docs/research/2026-10-14-finqa-data-pilot.md。